In [1]:
# ============================================================
# EpiGuard AI — Notebook 3: Risk Map & Dashboard Visuals
# ============================================================

# ── 0. GOOGLE DRIVE SETUP ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

BASE_PATH = "/content/drive/MyDrive/EpiGuard_AI"
PROCESSED_PATH = f"{BASE_PATH}/data/processed"
DASHBOARD_PATH = f"{BASE_PATH}/dashboard"

os.makedirs(DASHBOARD_PATH, exist_ok=True)

# ── 1. INSTALL + IMPORT LIBRARIES ────────────────────────────
try:
    import plotly.express as px
    import plotly.graph_objects as go
except:
    !pip install plotly -q
    import plotly.express as px
    import plotly.graph_objects as go

try:
    import pycountry
except:
    !pip install pycountry -q
    import pycountry

# ── 2. LOAD DATA FROM DRIVE ──────────────────────────────────
print("📂 Loading processed data from Drive...")

snapshot = pd.read_csv(f"{PROCESSED_PATH}/latest_snapshot.csv")
df = pd.read_csv(f"{PROCESSED_PATH}/processed_covid.csv", parse_dates=['date'])
forecast_df = pd.read_csv(f"{PROCESSED_PATH}/forecast_results.csv")

print(f"✅ Snapshot: {snapshot.shape}")
print(f"✅ Full data: {df.shape}")
print(f"✅ Forecast: {forecast_df.shape}")

# ── 3. ISO COUNTRY CODE FIXER ────────────────────────────────
iso_fix = {
    "US": "USA",
    "Russia": "RUS",
    "Korea, South": "KOR",
    "Iran": "IRN",
    "Vietnam": "VNM",
    "Bolivia": "BOL",
    "Syria": "SYR",
    "Laos": "LAO",
    "Moldova": "MDA",
}

def get_iso(name):
    if name in iso_fix:
        return iso_fix[name]
    try:
        return pycountry.countries.search_fuzzy(name)[0].alpha_3
    except:
        return None

snapshot["iso_code"] = snapshot["Country/Region"].apply(get_iso)
snapshot = snapshot.dropna(subset=["iso_code", "cases_per_million"])

# ── 4. CHOROPLETH RISK MAP ───────────────────────────────────
fig_map = px.choropleth(
    snapshot,
    locations="iso_code",
    color="cases_per_million",
    hover_name="Country/Region",
    hover_data={
        "cases_per_million": ":,.0f",
        "risk_tier": True,
        "confirmed": ":,.0f",
        "deaths": ":,.0f",
        "cfr": ":.2f"
    },
    color_continuous_scale="Reds",
    range_color=[0, snapshot["cases_per_million"].quantile(0.95)],
    title="🌍 EpiGuard AI — Global COVID Risk Map",
    labels={"cases_per_million": "Cases per Million"}
)

fig_map.update_layout(
    geo=dict(showframe=False, showcoastlines=True),
    height=550
)

fig_map.write_html(f"{DASHBOARD_PATH}/risk_map.html")
fig_map.show()

# ── 5. RISK DISTRIBUTION PIE ─────────────────────────────────
tier_counts = snapshot["risk_tier"].value_counts().reset_index()
tier_counts.columns = ["tier", "count"]

fig_pie = px.pie(
    tier_counts,
    names="tier",
    values="count",
    title="Global Risk Tier Distribution",
    color="tier",
    color_discrete_map={
        "Low": "#2ecc71",
        "Medium": "#f1c40f",
        "High": "#e67e22",
        "Critical": "#e74c3c"
    }
)

fig_pie.write_html(f"{DASHBOARD_PATH}/risk_distribution.html")
fig_pie.show()

# ── 6. TOP 10 TREND CHART ────────────────────────────────────
top10 = df.groupby("Country/Region")["confirmed"].max().nlargest(10).index
df_top = df[df["Country/Region"].isin(top10)]

fig_trend = go.Figure()

for country in top10:
    d = df_top[df_top["Country/Region"] == country]

    fig_trend.add_trace(go.Scatter(
        x=d["date"],
        y=d["rolling_7"],
        mode="lines",
        name=country
    ))

fig_trend.update_layout(
    title="📈 Top 10 Countries — 7-Day Rolling Trend",
    xaxis_title="Date",
    yaxis_title="Daily New Cases",
    hovermode="x unified",
    height=450
)

fig_trend.write_html(f"{DASHBOARD_PATH}/trend_top10.html")
fig_trend.show()

# ── 7. CASES VS DEATHS SCATTER ───────────────────────────────
fig_scatter = px.scatter(
    snapshot[snapshot["confirmed"] > 100000],
    x="confirmed",
    y="deaths",
    size="cases_per_million",
    color="risk_tier",
    hover_name="Country/Region",
    log_x=True,
    log_y=True,
    title="⚠️ Cases vs Deaths (High Burden Countries)"
)

fig_scatter.write_html(f"{DASHBOARD_PATH}/cases_vs_deaths.html")
fig_scatter.show()

# ── 8. SURGE ALERT TABLE FROM FORECASTS ──────────────────────
surge_summary = (
    forecast_df[forecast_df["alert"] == "Potential Surge"]
    .groupby("country")["yhat"]
    .mean()
    .sort_values(ascending=False)
    .reset_index()
)

surge_summary.columns = ["country", "avg_predicted_cases"]

surge_summary.to_csv(
    f"{DASHBOARD_PATH}/top_surge_countries.csv",
    index=False
)

print("\n🚨 Top predicted surge countries:")
print(surge_summary.head(10))

# ── 9. DASHBOARD SUMMARY EXPORT ──────────────────────────────
summary = {
    "total_countries": snapshot["Country/Region"].nunique(),
    "critical_count": (snapshot["risk_tier"] == "Critical").sum(),
    "high_count": (snapshot["risk_tier"] == "High").sum(),
    "surge_alert_countries": surge_summary["country"].nunique()
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv(f"{DASHBOARD_PATH}/dashboard_summary.csv", index=False)

print("\n✅ All dashboard assets saved to Google Drive")
print(f"📁 Dashboard folder: {DASHBOARD_PATH}")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 265.9 kB/s eta 0:00:00
📂 Loading processed data from Drive...
✅ Snapshot: (201, 12)
✅ Full data: (229743, 11)
✅ Forecast: (150, 6)



🚨 Top predicted surge countries:
  country  avg_predicted_cases
0  France         12126.438872
1  Brazil          9914.630923

✅ All dashboard assets saved to Google Drive
📁 Dashboard folder: /content/drive/MyDrive/EpiGuard_AI/dashboard
